# SpaceX Falcon 9 Data Collection with REST API

## IBM Applied Data Science Capstone

### Objective

Collect and prepare SpaceX launch data for analysis of Falcon 9 first-stage landing outcomes.

This notebook demonstrates:

- SpaceX REST API data collection
- JSON response processing and normalization
- launch-record preparation
- API enrichment methodology
- Falcon 9 dataset preparation
- missing-value analysis and handling

### Local validation note

During local validation, the live SpaceX API returned HTTP 522 responses. The notebook therefore documents the intended live REST API workflow and uses IBM Skills Network static course datasets where required to provide a reproducible analysis path.

In [1]:
import datetime

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

print("✓ Environment setup complete")
print(f"  requests: {requests.__version__}")
print(f"  pandas:   {pd.__version__}")
print(f"  numpy:    {np.__version__}")
print("  datetime: ready")


✓ Environment setup complete
  requests: 2.34.2
  pandas:   3.1.1
  numpy:    2.4.6
  datetime: ready


## 1. SpaceX REST API Data Collection

The IBM capstone collects historical SpaceX launch information using the SpaceX REST API. The intended launch endpoint is:

`https://api.spacexdata.com/v4/launches/past`

A successful request returns JSON launch records that can be normalized into a pandas DataFrame.

For reproducibility, this notebook first checks the live API. If the live service is unavailable, the analysis continues with the IBM Skills Network static copy of the SpaceX API response.

In [2]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"

live_response = requests.get(spacex_url, timeout=30)

print("SpaceX live API availability check")
print(f"  HTTP status:   {live_response.status_code}")
print(f"  Content-Type:  {live_response.headers.get('content-type')}")
print(f"  Response size: {len(live_response.content):,} bytes")

if live_response.status_code == 200:
    print("✓ Live SpaceX API is available")
else:
    print("⚠ Live SpaceX API is unavailable")
    print("  Continuing with IBM's static course dataset")

SpaceX live API availability check
  HTTP status:   522
  Content-Type:  text/html; charset=UTF-8
  Response size: 7,260 bytes
⚠ Live SpaceX API is unavailable
  Continuing with IBM's static course dataset


### Reproducible fallback: IBM Skills Network static API response

Because the live SpaceX REST API is unavailable in the current environment, the notebook continues with the IBM Skills Network static copy of the SpaceX API response.

This preserves the API-response structure required for the lab while allowing the remaining data-collection and preparation steps to execute reproducibly.

In [3]:
static_json_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
)

response = requests.get(static_json_url, timeout=30)
response.raise_for_status()

launches = response.json()

print("✓ IBM static SpaceX API response loaded")
print(f"  HTTP status:       {response.status_code}")
print(f"  Response type:     {type(launches).__name__}")
print(f"  Launch records:    {len(launches)}")
print(f"  First-record keys: {len(launches[0]) if launches else 0}")

✓ IBM static SpaceX API response loaded
  HTTP status:       200
  Response type:     list
  Launch records:    107
  First-record keys: 26


## 2. Normalize the JSON Launch Data

The API response contains nested JSON objects. `pandas.json_normalize()` converts these structured launch records into a flat DataFrame so the relevant launch fields can be selected and processed.

In [4]:
# Convert the JSON launch records into a pandas DataFrame
data = pd.json_normalize(launches)

print("✓ JSON normalized into DataFrame")
print(f"  Rows:    {data.shape[0]}")
print(f"  Columns: {data.shape[1]}")

data.head()

✓ JSON normalized into DataFrame
  Rows:    107
  Columns: 42


,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landin

## 3. Prepare the Launch Records

Before enriching the launch data, retain the fields needed for the analysis:

- rocket ID
- payload ID
- launchpad ID
- core information
- flight number
- launch date

Launches containing multiple cores or multiple payloads are excluded for this portion of the analysis. The remaining single core and payload IDs are extracted, and the launch date is converted to a Python date value.

In [5]:
# Retain the launch fields needed for the API data-collection workflow
data = data[
    ["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]
].copy()

# Keep launches with one core and one payload
data = data[data["cores"].map(len) == 1].copy()
data = data[data["payloads"].map(len) == 1].copy()

# Extract the single core and payload IDs
data["cores"] = data["cores"].map(lambda x: x[0])
data["payloads"] = data["payloads"].map(lambda x: x[0])

# Convert UTC launch timestamp to a date
data["date"] = pd.to_datetime(data["date_utc"]).dt.date

# Match the date range used by the IBM course lab
data = data[data["date"] <= datetime.date(2020, 11, 13)].copy()

print("✓ Launch records prepared")
print(f"  Records after filtering: {len(data)}")
print(f"  Columns retained:        {data.shape[1]}")
print(f"  Date range:              {data['date'].min()} to {data['date'].max()}")

data.head()

✓ Launch records prepared
  Records after filtering: 94
  Columns retained:        7
  Date range:              2006-03-24 to 2020-11-05


,rocket,payloads,launchpad,cores,flight_number,date_utc,date
0,5e9d0d95eda69955f709d1eb,5eb0e4b5b6c3bb0006eeb1e1,5e9e4502f5090995de566f86,"{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",1,2006-03-24T22:30:00.000Z,2006-03-24
1,5e9d0d95eda69955f709d1eb,5eb0e4b6b6c3bb0006eeb1e2,5e9e4502f5090995de566f86,"{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",2,2007-03-21T01:10:00.000Z,2007-03-21
3,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e5,5e9e4502f5090995de566f86,"{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",4,2008-09-28T23:15:00.000Z,2008-09-28
4,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e6,5e9e4502f5090995de566f86,"{'core': '5e9e289ef359184f103b2627', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",5,2009-07-13T03:35:00.000Z,2009-07-13
5,5e9d0d95eda69973a809d1ec,5eb0e4b7b6c3bb0006eeb1e7,5e9e4501f509094ba4566f84,"{'core': '5e9e289ef359185f2b3b2628', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",6,2010-06-04T18:45:00.000Z,2010-06-04


## 4. SpaceX API Enrichment Functions

The normalized launch records contain identifiers rather than all of the descriptive values needed for analysis.

The IBM lab uses additional SpaceX REST API endpoints to enrich the launch records with:

- booster version
- launch-site name and coordinates
- payload mass and orbit
- core reuse and landing information

The helper functions below document that intended REST API workflow.

> **Local validation note:** The live SpaceX enrichment endpoints returned HTTP 522 responses in this environment. The functions are therefore defined and retained as completed code, but they are not executed against all launch records. The notebook continues with IBM's prepared course dataset for reproducibility.

In [6]:
def get_booster_version(data, booster_versions):
    """Retrieve the booster name for each rocket ID."""
    for rocket_id in data["rocket"]:
        if rocket_id:
            response = requests.get(
                f"https://api.spacexdata.com/v4/rockets/{rocket_id}",
                timeout=30,
            )
            response.raise_for_status()
            booster_versions.append(response.json()["name"])


def get_launch_site(data, launch_sites, longitudes, latitudes):
    """Retrieve launch-site name and coordinates for each launchpad ID."""
    for launchpad_id in data["launchpad"]:
        if launchpad_id:
            response = requests.get(
                f"https://api.spacexdata.com/v4/launchpads/{launchpad_id}",
                timeout=30,
            )
            response.raise_for_status()

            launchpad = response.json()

            launch_sites.append(launchpad["name"])
            longitudes.append(launchpad["longitude"])
            latitudes.append(launchpad["latitude"])


def get_payload_data(data, payload_masses, orbits):
    """Retrieve payload mass and orbit for each payload ID."""
    for payload_id in data["payloads"]:
        if payload_id:
            response = requests.get(
                f"https://api.spacexdata.com/v4/payloads/{payload_id}",
                timeout=30,
            )
            response.raise_for_status()

            payload = response.json()

            payload_masses.append(payload.get("mass_kg"))
            orbits.append(payload.get("orbit"))


def get_core_data(
    data,
    outcomes,
    flights,
    grid_fins,
    reused,
    legs,
    landing_pads,
    blocks,
    reused_counts,
    serials,
):
    """Retrieve core and landing information."""
    for core in data["cores"]:
        if core["core"] is not None:
            response = requests.get(
                f"https://api.spacexdata.com/v4/cores/{core['core']}",
                timeout=30,
            )
            response.raise_for_status()

            core_info = response.json()

            blocks.append(core_info.get("block"))
            reused_counts.append(core_info.get("reuse_count"))
            serials.append(core_info.get("serial"))

        else:
            blocks.append(None)
            reused_counts.append(None)
            serials.append(None)

        outcomes.append(
            f"{core.get('landing_success')} {core.get('landing_type')}"
        )
        flights.append(core.get("flight"))
        grid_fins.append(core.get("gridfins"))
        reused.append(core.get("reused"))
        legs.append(core.get("legs"))
        landing_pads.append(core.get("landpad"))


print("✓ SpaceX API enrichment helper functions defined")
print("  Functions: booster, launch site, payload, core")

✓ SpaceX API enrichment helper functions defined
  Functions: booster, launch site, payload, core


## 5. Load the Prepared IBM Course Dataset

The live SpaceX enrichment endpoints are unavailable during local validation, so the helper functions above cannot be executed reliably against all launch records.

To continue the assignment reproducibly, the analysis uses IBM Skills Network's prepared `dataset_part_1.csv`. This dataset contains the enriched launch attributes required for the Falcon 9 filtering and missing-value tasks.

In [7]:
dataset_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
)

launch_data = pd.read_csv(dataset_url)

print("✓ IBM prepared launch dataset loaded")
print(f"  Rows:    {launch_data.shape[0]}")
print(f"  Columns: {launch_data.shape[1]}")

launch_data.head()

✓ IBM prepared launch dataset loaded
  Rows:    90
  Columns: 17


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 6. Task 1: Filter the Dataset for Falcon 9

The capstone analysis focuses on Falcon 9 launches. Any Falcon 1 records in the collected launch dataset must therefore be excluded.

The resulting Falcon 9 dataset is stored separately so the original prepared dataset remains unchanged.

In [8]:
# TASK 1
# Retain Falcon 9 launches by excluding Falcon 1 records
data_falcon9 = launch_data[
    launch_data["BoosterVersion"] != "Falcon 1"
].copy()

falcon1_removed = len(launch_data) - len(data_falcon9)

print("✓ TASK 1 completed")
print(f"  Original records:         {len(launch_data)}")
print(f"  Falcon 9 records:         {len(data_falcon9)}")
print(f"  Falcon 1 records removed: {falcon1_removed}")

data_falcon9.head()

✓ TASK 1 completed
  Original records:         90
  Falcon 9 records:         90
  Falcon 1 records removed: 0


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 7. Task 2: Reset the Falcon 9 Flight Numbers

After creating the Falcon 9 dataset, reset `FlightNumber` to a continuous sequence beginning with 1.

This provides a consistent launch sequence for the filtered dataset.

In [9]:
# TASK 2
# Reset FlightNumber to a continuous sequence
data_falcon9["FlightNumber"] = range(1, len(data_falcon9) + 1)

print("✓ TASK 2 completed")
print(f"  First flight number: {data_falcon9['FlightNumber'].iloc[0]}")
print(f"  Last flight number:  {data_falcon9['FlightNumber'].iloc[-1]}")
print(f"  Total records:       {len(data_falcon9)}")

data_falcon9.head()

✓ TASK 2 completed
  First flight number: 1
  Last flight number:  90
  Total records:       90


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 8. Missing-Value Analysis

Before handling missing payload-mass values, inspect the Falcon 9 dataset for null values.

`PayloadMass` null values will be handled in Task 3. `LandingPad` null values are retained because a null landing pad can represent a launch where a landing pad was not used.

In [10]:
# Inspect missing values before Task 3
missing_values = data_falcon9.isnull().sum()

print("✓ Missing-value analysis completed")
print(f"  PayloadMass missing: {missing_values['PayloadMass']}")
print(f"  LandingPad missing:  {missing_values['LandingPad']}")

missing_values

✓ Missing-value analysis completed
  PayloadMass missing: 0
  LandingPad missing:  26


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

## 9. Task 3: Handle Missing Payload Mass Values

The IBM lab requires calculating the mean `PayloadMass` and replacing missing payload-mass values with that mean.

In the prepared course dataset used for this reproducible fallback, `PayloadMass` currently contains no missing values. The required imputation logic is still applied so the workflow remains valid if missing values are present.

`LandingPad` null values are intentionally retained because they can represent launches where a landing pad was not used.

In [11]:
# TASK 3
# Calculate the mean PayloadMass
payload_mean = data_falcon9["PayloadMass"].mean()

# Record missing values before imputation
payload_missing_before = data_falcon9["PayloadMass"].isnull().sum()

# Replace missing PayloadMass values with the mean
data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].fillna(
    payload_mean
)

# Validate the result
payload_missing_after = data_falcon9["PayloadMass"].isnull().sum()
landingpad_missing = data_falcon9["LandingPad"].isnull().sum()

print("✓ TASK 3 completed")
print(f"  Mean PayloadMass:       {payload_mean:.2f} kg")
print(f"  Missing values before:  {payload_missing_before}")
print(f"  Missing values after:   {payload_missing_after}")
print(f"  LandingPad nulls kept:  {landingpad_missing}")

data_falcon9.isnull().sum()

✓ TASK 3 completed
  Mean PayloadMass:       6104.96 kg
  Missing values before:  0
  Missing values after:   0
  LandingPad nulls kept:  26


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

## 10. Final Validation Summary

The SpaceX API data-collection workflow has been completed and validated locally.

### Completed workflow

- Configured the Python analysis environment.
- Checked the live SpaceX REST API endpoint.
- Documented the HTTP 522 condition encountered during local validation.
- Loaded IBM Skills Network's static SpaceX API response for reproducibility.
- Parsed and normalized the JSON launch records into a pandas DataFrame.
- Prepared the launch records required for API enrichment.
- Defined helper functions for booster, launch-site, payload, and core enrichment.
- Loaded IBM's prepared enriched launch dataset when the live enrichment endpoints were unavailable.
- Filtered the analysis dataset for Falcon 9 launches.
- Reset Falcon 9 flight numbers.
- Inspected missing values.
- Calculated mean PayloadMass and applied the required missing-value handling.
- Preserved LandingPad null values as required.

The resulting Falcon 9 dataset is ready for subsequent capstone analysis.

In [12]:
# Final validation summary
final_missing = data_falcon9.isnull().sum()

print("✓ Notebook 01 validation complete")
print(f"  Final records:              {len(data_falcon9)}")
print(f"  Final columns:              {data_falcon9.shape[1]}")
print(f"  Flight-number range:        {data_falcon9['FlightNumber'].min()}-{data_falcon9['FlightNumber'].max()}")
print(f"  PayloadMass null values:    {final_missing['PayloadMass']}")
print(f"  LandingPad null values:     {final_missing['LandingPad']}")
print(f"  Mean PayloadMass:           {data_falcon9['PayloadMass'].mean():.2f} kg")

✓ Notebook 01 validation complete
  Final records:              90
  Final columns:              17
  Flight-number range:        1-90
  PayloadMass null values:    0
  LandingPad null values:     26
  Mean PayloadMass:           6104.96 kg
